# FIFA World Cup 2026: Tournament Optimisation

Three linked integer programming models (Pyomo + GLPK): balanced group formation, group-letter assignment, and stadium allocation. See the [README](../README.md) for the full write-up.

**Notes for readers**
- Data is loaded from `../data/FIFA_data.xlsx`. GLPK must be installed (`conda install -c conda-forge glpk`).
- The Model 1 solver log shown below is from an earlier run made before the final data file. The final optimised groups (hard-coded at the start of Models 2 and 3) come from the run reported in the README, with a weakest-group total of 6,335.30.
- Known data issues: the baseline points list in Model 1 uses 1,160.00 for New Zealand, while the data file uses 1,279.25. Iraq's GDP per capita in the data file (71,026.5) duplicates Denmark's; the correct figure is about 4,900. See *Limitations* in the README.

In [1]:
# Model 1: Balanced group formation
import os, shutil
import pandas as pd
from pyomo.environ import *
import matplotlib.pyplot as plt
import numpy as np

file_name = "../data/FIFA_data.xlsx"  
sheet = "Team-Level Data"
df = pd.read_excel(file_name, sheet_name=sheet)
df["Team_Name"] = df["Team_Name"].astype(str)
df["FIFA_Points"] = pd.to_numeric(df["FIFA_Points"], errors="coerce")
df["Pot"] = pd.to_numeric(df["Pot"], errors="coerce")


# Fill the 6 placeholders using doc proxies
for idx in range(len(df)):
    name = df.loc[idx, "Team_Name"]
    if pd.isna(df.loc[idx, "FIFA_Points"]):

        if name.startswith("Winner play-off A"):
            df.loc[idx, "Confederation"] = "UEFA"
            df.loc[idx, "FIFA_Points"] = 1702.06   # UEFA Path A Italy proxy [file:44]
        elif name.startswith("Winner play-off B"):
            df.loc[idx, "Confederation"] = "UEFA"
            df.loc[idx, "FIFA_Points"] = 1557.47   # UEFA Path B Ukraine proxy [file:44]
        elif name.startswith("Winner play-off C"):
            df.loc[idx, "Confederation"] = "UEFA"
            df.loc[idx, "FIFA_Points"] = 1582.69   # UEFA Path C Türkiye proxy [file:44]
        elif name.startswith("Winner play-off D"):
            df.loc[idx, "Confederation"] = "UEFA"
            df.loc[idx, "FIFA_Points"] = 1616.75   # UEFA Path D Denmark proxy [file:44]
        elif name.startswith("Winner play-off 1"):
            df.loc[idx, "Confederation"] = "CAF"
            df.loc[idx, "FIFA_Points"] = 1468.22   # IC Path 1 DR Congo proxy [file:44]
        elif name.startswith("Winner play-off 2"):
            df.loc[idx, "Confederation"] = "AFC"
            df.loc[idx, "FIFA_Points"] = 1436.94   # IC Path 2 Iraq proxy [file:44]

# Check the dataset
## ROWS
print( df.shape[0])

##Missing FIFA_Points:
print(df["FIFA_Points"].isna().sum())

##Missing Confederation:
print( df["Confederation"].isna().sum())

#  Prepare sets/parameters
teams = df["Team_Name"].tolist()

points = dict(zip(df["Team_Name"], df["FIFA_Points"]))
confed = dict(zip(df["Team_Name"], df["Confederation"]))
pot_of = dict(zip(df["Team_Name"], df["Pot"]))
baseline_group = dict(zip(df["Team_Name"], df["Group"]))

numGroups = 12
pots = sorted(df["Pot"].unique().tolist())
confeds = sorted(df["Confederation"].unique().tolist())

# UEFA <= 2, others <= 1 (Version 1)
cap = {}
for c in confeds:
    if c == "UEFA":
        cap[c] = 2
    else:
        cap[c] = 1

# quick sanity check: each pot should have 12 teams
for k in pots:
    count_k = sum(1 for t in teams if int(pot_of[t]) == int(k))
    print("Pot", int(k), "count =", count_k)

model = ConcreteModel()
model.x = Var(range(len(teams)), range(numGroups), domain=Binary)
model.w = Var(domain=NonNegativeReals)


# build index sets once
pot_idx = {k: [i for i,t in enumerate(teams) if int(pot_of[t])==int(k)] for k in pots}
conf_idx = {c: [i for i,t in enumerate(teams) if confed[t]==c] for c in confeds}

def obj_rule(model):
    return model.w
model.obj = Objective(rule=obj_rule, sense=maximize)

## w ≤ Total score of each group
##Make the weakest group as strong as possible, thereby making the groupings more balanced
def min_points_rule(model, g):
    return model.w <= sum(points[teams[i]] * model.x[i, g] for i in range(len(teams)))
model.min_points = Constraint(range(numGroups), rule=min_points_rule)

## Each group g must have exactly 4 teams.
def group_size_rule(model, g):
    return sum(model.x[i, g] for i in range(len(teams))) == 4
model.group_size = Constraint(range(numGroups), rule=group_size_rule)

## Each team i can only be placed in one group, and must be placed in a group.
def assign_once_rule(model, i):
    return sum(model.x[i, g] for g in range(numGroups)) == 1
model.assign_once = Constraint(range(len(teams)), rule=assign_once_rule)

## Group g: From each pot k, exactly one team must be selected.
def pot_rule(model, g, k_index):
    k = pots[k_index]
    return sum(model.x[i, g] for i in pot_idx[k]) == 1
model.pot_cons = Constraint(range(numGroups), range(len(pots)), rule=pot_rule)

## UEFA can have at most 2 teams, and for others, it can have at most 1 team.
def confed_rule(model, g, c_index):
    c = confeds[c_index]
    return sum(model.x[i, g] for i in conf_idx[c]) <= cap[c]
model.confed_cons = Constraint(range(numGroups), range(len(confeds)), rule=confed_rule)


solver = SolverFactory("glpk")
solver.options["tmlim"] = 120
results = solver.solve(model, tee=True)

## print result
print(value(model.w))

##Calculate the point value of the baseline
points = {
    # Pot 1
    "Spain": 1877.18, "Argentina": 1873.33, "France": 1870.00, "England": 1834.12,
    "Brazil": 1760.46, "Portugal": 1760.38, "Netherlands": 1756.27, "Belgium": 1730.71,
    "Germany": 1724.15, "USA": 1681.88, "Mexico": 1675.75, "Canada": 1559.15,

    # Pot 2 / 3 / 4
    "Morocco": 1736.57, "Croatia": 1716.88, "Senegal": 1706.83, "Colombia": 1701.30,
    "Uruguay": 1672.62, "Switzerland": 1654.69, "Japan": 1650.12, "IR Iran": 1617.02,
    "Iraq": 1617.02, "Egypt": 1556.71, "Paraguay": 1501.50, "Scotland": 1506.77,
    "Tunisia": 1479.04, "Uzbekistan": 1462.03, "Jordan": 1388.93, "Cabo Verde": 1370.49,
    "Ghana": 1351.09, "New Zealand": 1160.00, "Ecuador": 1591.73, "Curaçao": 1302.70,
    "Curacao": 1302.70, "Cte dIvoire": 1522.48, "Ivory Coast": 1522.48,
    "South Africa": 1432.76, "Korea Republic": 1599.45, "Qatar": 1454.96,
    "Saudi Arabia": 1429.48, "Haiti": 1294.49, "Panama": 1539.47, "Australia": 1574.01,
    "Algeria": 1560.91, "Austria": 1585.51, "Norway": 1553.14,

    # Proxy points
    "Winner play-off A": 1702.06,
    "Winner play-off B": 1557.47,
    "Winner play-off C": 1582.69,
    "Winner play-off D": 1616.75,
    "Winner play-off 1": 1468.22,
    "Winner play-off 2": 1436.94
}

groups = {
    "A": ["Mexico", "South Africa", "Korea Republic", "Winner play-off D"],
    "B": ["Canada", "Winner play-off A", "Qatar", "Switzerland"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["USA", "Paraguay", "Australia", "Winner play-off C"],
    "E": ["Germany", "Curacao", "Cte dIvoire", "Ecuador"],
    "F": ["Netherlands", "Japan", "Winner play-off B", "Tunisia"],
    "G": ["Belgium", "Egypt", "IR Iran", "New Zealand"],
    "H": ["Spain", "Cabo Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Winner play-off 2", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "Winner play-off 1", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"]
}
baseline_df = pd.DataFrame({
    "Group": list(groups.keys()),
    "BaselineTotalPts": [sum(points[t] for t in groups[g]) for g in groups]
}).sort_values("Group")

print(baseline_df.to_string(index=False))
baseline_df.to_csv("baseline_totals.csv", index=False)
# baseline_df:  Group, BaselineTotalPts
df = baseline_df.copy()
df["BaselineTotalPts"] = pd.to_numeric(df["BaselineTotalPts"], errors="coerce")

# 1) 
stats = { "min": df["BaselineTotalPts"].min(), "max": df["BaselineTotalPts"].max(),"range": df["BaselineTotalPts"].max() - df["BaselineTotalPts"].min(),"mean": df["BaselineTotalPts"].mean(),"std": df["BaselineTotalPts"].std(ddof=1),"cv(std/mean)": df["BaselineTotalPts"].std(ddof=1) / df["BaselineTotalPts"].mean(),
 }
print(pd.Series(stats).round(2).to_string())

# 2) 
ranked = df.sort_values("BaselineTotalPts").reset_index(drop=True)
print("\nBaseline groups (low -> high):")
print(ranked.to_string(index=False))

48
0
0
Pot 1 count = 12
Pot 2 count = 12
Pot 3 count = 12
Pot 4 count = 12
GLPSOL: GLPK LP/MIP Solver, v4.65
Parameter(s) specified in the command line:
 --tmlim 120 --write C:\Users\ASUS\AppData\Local\Temp\tmp3w923m02.glpk.raw
 --wglp C:\Users\ASUS\AppData\Local\Temp\tmpbyhxxxqh.glpk.glp --cpxlp C:\Users\ASUS\AppData\Local\Temp\tmphmve_f2c.pyomo.lp
Reading problem data from 'C:\Users\ASUS\AppData\Local\Temp\tmphmve_f2c.pyomo.lp'...
C:\Users\ASUS\AppData\Local\Temp\tmphmve_f2c.pyomo.lp:4056: warning: lower bound of variable 'x4' redefined
C:\Users\ASUS\AppData\Local\Temp\tmphmve_f2c.pyomo.lp:4056: warning: upper bound of variable 'x4' redefined
192 rows, 577 columns, 2892 non-zeros
576 integer variables, all of which are binary
4632 lines were read
Writing problem data to 'C:\Users\ASUS\AppData\Local\Temp\tmpbyhxxxqh.glpk.glp'...
3858 lines were written
GLPK Integer Optimizer, v4.65
192 rows, 577 columns, 2892 non-zeros
576 integer variables, all of which are binary
Preprocessing...
18

In [1]:
# Model 2: Group-letter assignment (balancing spectator demand across the schedule)
# 2026 FIFA World Cup
# Final version used for the report

import pandas as pd
from pyomo.environ import (
    ConcreteModel, Set, Var, Binary, NonNegativeReals,
    Objective, Constraint, ConstraintList, SolverFactory,
    maximize, value
)
from pyomo.opt import SolverStatus, TerminationCondition


# Read the team-level data and make sure GDP is numeric
file_name =  "../data/FIFA_data.xlsx"  
teams = pd.read_excel(file_name, sheet_name="Team-Level Data")
teams.columns = teams.columns.str.strip()
teams["gdp_per_capita($)"] = pd.to_numeric(teams["gdp_per_capita($)"], errors="coerce")


# These are the optimized groups coming from Question 2
optimized_groups = {
    1: ["Argentina", "Morocco", "Uzbekistan", "New Zealand"],
    2: ["Germany", "Paraguay", "Qatar", "Switzerland"],
    3: ["Spain", "Cabo Verde", "Saudi Arabia", "Uruguay"],
    4: ["England", "Algeria", "Iraq", "Panama"],
    5: ["Brazil", "South Africa", "Ukraine", "Denmark"],
    6: ["Canada", "Italy", "Korea Republic", "Tunisia"],
    7: ["Portugal", "Curaçao", "Austria", "Colombia"],
    8: ["France", "Japan", "Ghana", "Scotland"],
    9: ["Netherlands", "Senegal", "Haiti", "Turkey"],
    10: ["Mexico", "Egypt", "Australia", "Norway"],
    11: ["Belgium", "Croatia", "Côte d'Ivoire", "Jordan"],
    12: ["USA", "DR Congo", "IR Iran", "Ecuador"],
}


# GDP per capita is used here as the supporter-demand proxy
gdp_map = dict(zip(teams["Team_Name"], teams["gdp_per_capita($)"]))

# A few teams needed manual values because of missing or mismatched entries
manual_gdp = {
    "Iraq": 4900,
    "Ukraine": 4800,
    "Denmark": 68000,
    "Italy": 34000,
    "Turkey": 13000,
    "DR Congo": 600,
}
for team, gdp in manual_gdp.items():
    if team not in gdp_map or pd.isna(gdp_map.get(team)):
        gdp_map[team] = gdp


# Stadium capacities for the 2026 host venues used in the group stage
stadium_capacity_map = {
    "Mexico City": 72766,
    "Guadalajara": 44330,
    "Toronto": 44315,
    "Los Angeles": 69650,
    "San Francisco Bay Area": 69391,
    "New York New Jersey": 78576,
    "Boston": 63815,
    "Vancouver": 54500,
    "Houston": 68311,
    "Dallas": 70122,
    "Philadelphia": 65827,
    "Monterrey": 50113,
    "Atlanta": 67382,
    "Seattle": 65123,
    "Miami": 64091,
    "Kansas City": 67513,
}


# Spectator index from the Ghoniem-style setup
# offered_seats is the number of seats available to each competing nation
officials_share = 0.09
team_ticket_share = 0.12
offered_seats = [
    (1 - officials_share) * team_ticket_share * cap
    for cap in stadium_capacity_map.values()
]

def spectator_index(gdp_value):
    if pd.isna(gdp_value) or gdp_value == 0:
        return 0.0
    avg_ratio = sum((1 / len(offered_seats)) * (gdp_value / seats) for seats in offered_seats)
    return min(1.0, avg_ratio)

spectator_index_map = {
    team: spectator_index(gdp_value)
    for team, gdp_value in gdp_map.items()
    if pd.notna(gdp_value)
}


# Match popularity follows the same structure used in the report
def match_popularity(team_1, team_2):
    fan_share_1 = spectator_index_map.get(team_1, 0.0)
    fan_share_2 = spectator_index_map.get(team_2, 0.0)
    other_nations = (fan_share_1 + fan_share_2) / 2
    officials = 1.0 if max(fan_share_1, fan_share_2) == 1.0 else (fan_share_1 + fan_share_2) / 2
    return fan_share_1 + fan_share_2 + other_nations + officials


# Six matches per four-team group
match_slots = {
    1: (0, 1),
    2: (2, 3),
    3: (0, 2),
    4: (1, 3),
    5: (0, 3),
    6: (1, 2),
}

group_match_popularity = {}
group_total_popularity = {}
for group_id, teams_in_group in optimized_groups.items():
    group_match_popularity[group_id] = {}
    total_popularity = 0.0
    for match_number, (i, j) in match_slots.items():
        popularity_value = match_popularity(teams_in_group[i], teams_in_group[j])
        group_match_popularity[group_id][match_number] = popularity_value
        total_popularity += popularity_value
    group_total_popularity[group_id] = total_popularity


# Corrected 2026 group-stage schedule
# Each row is one scheduled match slot: date, stadium, group letter, and match number inside the group
raw_schedule = [
    ["11-Jun-2026", "Mexico City", "A", 1],
    ["12-Jun-2026", "Guadalajara", "A", 2],
    ["12-Jun-2026", "Toronto", "B", 1],
    ["13-Jun-2026", "Los Angeles", "D", 1],
    ["13-Jun-2026", "San Francisco Bay Area", "B", 2],
    ["13-Jun-2026", "New York New Jersey", "C", 1],
    ["14-Jun-2026", "Boston", "C", 2],
    ["14-Jun-2026", "Vancouver", "D", 2],
    ["14-Jun-2026", "Houston", "E", 1],
    ["14-Jun-2026", "Dallas", "F", 1],
    ["15-Jun-2026", "Philadelphia", "E", 2],
    ["15-Jun-2026", "Monterrey", "F", 2],
    ["15-Jun-2026", "Atlanta", "H", 1],
    ["15-Jun-2026", "Seattle", "G", 1],
    ["15-Jun-2026", "Miami", "H", 2],
    ["16-Jun-2026", "Los Angeles", "G", 2],
    ["16-Jun-2026", "New York New Jersey", "I", 1],
    ["16-Jun-2026", "Boston", "I", 2],
    ["17-Jun-2026", "Kansas City", "J", 1],
    ["17-Jun-2026", "San Francisco Bay Area", "J", 2],
    ["17-Jun-2026", "Houston", "K", 1],
    ["17-Jun-2026", "Dallas", "L", 1],
    ["18-Jun-2026", "Toronto", "L", 2],
    ["18-Jun-2026", "Mexico City", "K", 2],
    ["18-Jun-2026", "Atlanta", "A", 3],
    ["18-Jun-2026", "Los Angeles", "B", 3],
    ["18-Jun-2026", "Vancouver", "B", 4],
    ["19-Jun-2026", "Guadalajara", "A", 4],
    ["19-Jun-2026", "Seattle", "D", 3],
    ["19-Jun-2026", "Boston", "C", 3],
    ["20-Jun-2026", "Philadelphia", "C", 4],
    ["20-Jun-2026", "San Francisco Bay Area", "D", 4],
    ["20-Jun-2026", "Houston", "F", 3],
    ["20-Jun-2026", "Toronto", "E", 3],
    ["21-Jun-2026", "Kansas City", "E", 4],
    ["21-Jun-2026", "Monterrey", "F", 4],
    ["21-Jun-2026", "Atlanta", "H", 3],
    ["21-Jun-2026", "Los Angeles", "G", 3],
    ["21-Jun-2026", "Miami", "H", 4],
    ["21-Jun-2026", "Vancouver", "G", 4],
    ["22-Jun-2026", "Dallas", "J", 3],
    ["22-Jun-2026", "Philadelphia", "I", 3],
    ["23-Jun-2026", "New York New Jersey", "I", 4],
    ["23-Jun-2026", "San Francisco Bay Area", "J", 4],
    ["23-Jun-2026", "Houston", "K", 3],
    ["23-Jun-2026", "Boston", "L", 3],
    ["24-Jun-2026", "Toronto", "L", 4],
    ["24-Jun-2026", "Guadalajara", "K", 4],
    ["24-Jun-2026", "Vancouver", "B", 5],
    ["24-Jun-2026", "Seattle", "B", 6],
    ["24-Jun-2026", "Miami", "C", 5],
    ["24-Jun-2026", "Atlanta", "C", 6],
    ["25-Jun-2026", "Mexico City", "A", 5],
    ["25-Jun-2026", "Monterrey", "A", 6],
    ["25-Jun-2026", "Philadelphia", "E", 5],
    ["25-Jun-2026", "New York New Jersey", "E", 6],
    ["26-Jun-2026", "Dallas", "F", 5],
    ["26-Jun-2026", "Kansas City", "F", 6],
    ["26-Jun-2026", "Los Angeles", "D", 5],
    ["26-Jun-2026", "San Francisco Bay Area", "D", 6],
    ["26-Jun-2026", "Boston", "I", 5],
    ["26-Jun-2026", "Toronto", "I", 6],
    ["27-Jun-2026", "Houston", "H", 5],
    ["27-Jun-2026", "Guadalajara", "H", 6],
    ["27-Jun-2026", "Seattle", "G", 5],
    ["27-Jun-2026", "Vancouver", "G", 6],
    ["27-Jun-2026", "New York New Jersey", "L", 5],
    ["27-Jun-2026", "Philadelphia", "L", 6],
    ["28-Jun-2026", "Miami", "K", 5],
    ["28-Jun-2026", "Atlanta", "K", 6],
    ["28-Jun-2026", "Kansas City", "J", 5],
    ["28-Jun-2026", "Dallas", "J", 6],
]

schedule = pd.DataFrame(raw_schedule, columns=["Date", "Stadium", "Letter", "MatchInGroup"])
schedule = schedule.drop_duplicates().reset_index(drop=True)
schedule["RowSet"] = schedule["Date"] + " | " + schedule["Stadium"]

group_ids = list(optimized_groups.keys())
group_letters = sorted(schedule["Letter"].unique())
rowsets = list(schedule["RowSet"].unique())


# For each row-set, keep track of which group letter and match slot feed into it
rowset_match_map = {}
for rowset in rowsets:
    rowset_rows = schedule[schedule["RowSet"] == rowset][["Letter", "MatchInGroup"]].values.tolist()
    rowset_match_map[rowset] = [(row[0], int(row[1])) for row in rowset_rows]


# Main model:
# assign each optimized group to one letter
# then compute the induced row-set popularities
# objective is to push up the worst row-set, with a tiny tie-break on total popularity
tie_break_weight = 0.01

assignment_model = ConcreteModel()
assignment_model.groups = Set(initialize=group_ids)
assignment_model.letters = Set(initialize=group_letters)
assignment_model.rowsets = Set(initialize=rowsets)

assignment_model.assign = Var(assignment_model.groups, assignment_model.letters, domain=Binary)
assignment_model.rowset_popularity = Var(assignment_model.rowsets, domain=NonNegativeReals)
assignment_model.min_rowset_popularity = Var(domain=NonNegativeReals)

assignment_model.objective = Objective(
    expr=assignment_model.min_rowset_popularity
    + tie_break_weight * sum(assignment_model.rowset_popularity[rowset] for rowset in assignment_model.rowsets),
    sense=maximize
)

def one_group_per_letter_rule(model, letter):
    return sum(model.assign[group_id, letter] for group_id in model.groups) == 1

assignment_model.one_group_per_letter = Constraint(
    assignment_model.letters, rule=one_group_per_letter_rule
)

def one_letter_per_group_rule(model, group_id):
    return sum(model.assign[group_id, letter] for letter in model.letters) == 1

assignment_model.one_letter_per_group = Constraint(
    assignment_model.groups, rule=one_letter_per_group_rule
)

# Host countries are fixed to the required letters
assignment_model.host_mexico = Constraint(expr=assignment_model.assign[10, "A"] == 1)
assignment_model.host_canada = Constraint(expr=assignment_model.assign[6, "B"] == 1)
assignment_model.host_usa = Constraint(expr=assignment_model.assign[12, "D"] == 1)

# This links each row-set popularity to the group assigned to its letter
assignment_model.rowset_popularity_definition = ConstraintList()
for rowset in rowsets:
    rowset_expression = 0
    for letter, match_number in rowset_match_map[rowset]:
        rowset_expression += sum(
            group_match_popularity[group_id][match_number] * assignment_model.assign[group_id, letter]
            for group_id in group_ids
        )
    assignment_model.rowset_popularity_definition.add(
        assignment_model.rowset_popularity[rowset] == rowset_expression
    )

def min_rowset_rule(model, rowset):
    return model.min_rowset_popularity <= model.rowset_popularity[rowset]

assignment_model.min_rowset_constraint = Constraint(
    assignment_model.rowsets, rule=min_rowset_rule
)


# Solve with GLPK
solver = SolverFactory("glpk")
result = solver.solve(assignment_model, tee=False)

print("=" * 60)
print("SOLVER STATUS")
print("=" * 60)
print("Status:", result.solver.status)
print("Termination condition:", result.solver.termination_condition)

if result.solver.status != SolverStatus.ok:
    raise RuntimeError("Solver did not return status ok.")

if result.solver.termination_condition not in [TerminationCondition.optimal, TerminationCondition.feasible]:
    raise RuntimeError("Solver did not return a feasible or optimal solution.")


# Pull the chosen group-letter assignment out of the model
group_letter_assignment = {}
for group_id in group_ids:
    for letter in group_letters:
        if value(assignment_model.assign[group_id, letter]) > 0.5:
            group_letter_assignment[group_id] = letter

assignment_df = (
    pd.DataFrame({
        "Optimized_Group": list(group_letter_assignment.keys()),
        "Assigned_Letter": list(group_letter_assignment.values()),
        "Teams": [", ".join(optimized_groups[group_id]) for group_id in group_letter_assignment.keys()]
    })
    .sort_values("Assigned_Letter")
    .reset_index(drop=True)
)

letter_to_group = {letter: group_id for group_id, letter in group_letter_assignment.items()}


# Rebuild the schedule using the assigned groups so row-set popularity can be calculated directly
schedule["Optimized_Group"] = schedule["Letter"].map(letter_to_group)
schedule["MatchPopularity"] = schedule.apply(
    lambda row: group_match_popularity[row["Optimized_Group"]][row["MatchInGroup"]],
    axis=1
)

rowset_popularity = (
    schedule.groupby(["RowSet", "Date", "Stadium"], as_index=False)["MatchPopularity"]
    .sum()
    .sort_values("MatchPopularity", ascending=False)
    .reset_index(drop=True)
)

min_rowset_popularity = value(assignment_model.min_rowset_popularity)
max_rowset_popularity = rowset_popularity["MatchPopularity"].max()
mean_rowset_popularity = rowset_popularity["MatchPopularity"].mean()


# Final outputs used in the write-up
print("\n" + "=" * 60)
print("GROUP-LETTER ASSIGNMENT")
print("=" * 60)
print(assignment_df.to_string(index=False))

print("\n" + "=" * 60)
print("ROW-SET POPULARITY SUMMARY")
print("=" * 60)
print(f"Maximum row-set popularity (realized): {max_rowset_popularity:.4f}")
print(f"Minimum row-set popularity (optimized): {min_rowset_popularity:.4f}")
print(f"Mean row-set popularity: {mean_rowset_popularity:.4f}")
print(f"Number of row-sets: {len(rowset_popularity)}")

print("\nTop 5 row-sets by popularity:")
print(rowset_popularity.head(5).to_string(index=False))

print("\nBottom 5 row-sets by popularity:")
print(rowset_popularity.tail(5).to_string(index=False))


SOLVER STATUS
Status: ok
Termination condition: optimal

GROUP-LETTER ASSIGNMENT
 Optimized_Group Assigned_Letter                                       Teams
              10               A            Mexico, Egypt, Australia, Norway
               6               B      Canada, Italy, Korea Republic, Tunisia
               1               C Argentina, Morocco, Uzbekistan, New Zealand
              12               D             USA, DR Congo, IR Iran, Ecuador
               2               E       Germany, Paraguay, Qatar, Switzerland
               3               F    Spain, Cabo Verde, Saudi Arabia, Uruguay
               4               G              England, Algeria, Iraq, Panama
               5               H      Brazil, South Africa, Ukraine, Denmark
               7               I        Portugal, Curaçao, Austria, Colombia
               8               J              France, Japan, Ghana, Scotland
              11               K     Belgium, Croatia, Côte d'Ivoire, Jo

In [13]:
# Model 3: Stadium assignment (matching demand to capacity)
# 2026 FIFA World Cup
# Final version based on the corrected Q3 setup

import pandas as pd
from pyomo.environ import (
    ConcreteModel, Set, Var, Binary, Param,
    Objective, ConstraintList, SolverFactory,
    maximize, value
)
from pyomo.opt import SolverStatus, TerminationCondition


# Read the team data and clean up the GDP column
file_name = "../data/FIFA_data.xlsx"
teams = pd.read_excel(file_name, sheet_name="Team-Level Data")
teams.columns = teams.columns.str.strip()
teams["gdp_per_capita($)"] = pd.to_numeric(teams["gdp_per_capita($)"], errors="coerce")


# These are the optimized groups coming from Q2
optimized_groups = {
    1: ["Argentina", "Morocco", "Uzbekistan", "New Zealand"],
    2: ["Germany", "Paraguay", "Qatar", "Switzerland"],
    3: ["Spain", "Cabo Verde", "Saudi Arabia", "Uruguay"],
    4: ["England", "Algeria", "Iraq", "Panama"],
    5: ["Brazil", "South Africa", "Ukraine", "Denmark"],
    6: ["Canada", "Italy", "Korea Republic", "Tunisia"],
    7: ["Portugal", "Curaçao", "Austria", "Colombia"],
    8: ["France", "Japan", "Ghana", "Scotland"],
    9: ["Netherlands", "Senegal", "Haiti", "Turkey"],
    10: ["Mexico", "Egypt", "Australia", "Norway"],
    11: ["Belgium", "Croatia", "Côte d'Ivoire", "Jordan"],
    12: ["USA", "DR Congo", "IR Iran", "Ecuador"],
}


# GDP per capita is used here as the supporter-demand proxy
gdp_map = dict(zip(teams["Team_Name"], teams["gdp_per_capita($)"]))

# A few teams needed manual values because of missing or inconsistent entries
manual_gdp = {
    "Iraq": 4900,
    "Ukraine": 4800,
    "Denmark": 68000,
    "Italy": 34000,
    "Turkey": 13000,
    "DR Congo": 600,
}
for team, gdp in manual_gdp.items():
    if team not in gdp_map or pd.isna(gdp_map.get(team)):
        gdp_map[team] = gdp


# Group-stage stadium capacities used in the model
stadium_capacity_map = {
    "Mexico City": 72766,
    "Guadalajara": 44330,
    "Toronto": 44315,
    "Los Angeles": 69650,
    "San Francisco Bay Area": 69391,
    "New York New Jersey": 78576,
    "Boston": 63815,
    "Vancouver": 54500,
    "Houston": 68311,
    "Dallas": 70122,
    "Philadelphia": 65827,
    "Monterrey": 50113,
    "Atlanta": 67382,
    "Seattle": 65123,
    "Miami": 64091,
    "Kansas City": 67513,
}


# Spectator index from the same setup used in Q3
officials_share = 0.09
team_ticket_share = 0.12
offered_seats = [(1 - officials_share) * team_ticket_share * cap for cap in stadium_capacity_map.values()]

def spectator_index(gdp_value):
    if pd.isna(gdp_value) or gdp_value == 0:
        return 0.0
    avg_ratio = sum((1 / len(offered_seats)) * (gdp_value / seats) for seats in offered_seats)
    return min(1.0, avg_ratio)

spectator_index_map = {
    team: spectator_index(gdp_value)
    for team, gdp_value in gdp_map.items()
    if pd.notna(gdp_value)
}


# Match popularity uses the same definition carried over from Q3
def match_popularity(team_1, team_2):
    fan_share_1 = spectator_index_map.get(team_1, 0.0)
    fan_share_2 = spectator_index_map.get(team_2, 0.0)
    other_nations = (fan_share_1 + fan_share_2) / 2
    officials = 1.0 if max(fan_share_1, fan_share_2) == 1.0 else (fan_share_1 + fan_share_2) / 2
    return fan_share_1 + fan_share_2 + other_nations + officials


# Six matches inside each four-team group
match_slots = {
    1: (0, 1),
    2: (2, 3),
    3: (0, 2),
    4: (1, 3),
    5: (0, 3),
    6: (1, 2),
}

group_match_popularity = {}
for group_id, teams_in_group in optimized_groups.items():
    group_match_popularity[group_id] = {}
    for match_number, (i, j) in match_slots.items():
        group_match_popularity[group_id][match_number] = match_popularity(
            teams_in_group[i], teams_in_group[j]
        )


# This is the corrected group-stage schedule carried over from final Q3
raw_schedule = [
    ["11-Jun-2026", "Mexico City", "A", 1],
    ["12-Jun-2026", "Guadalajara", "A", 2],
    ["12-Jun-2026", "Toronto", "B", 1],
    ["13-Jun-2026", "Los Angeles", "D", 1],
    ["13-Jun-2026", "San Francisco Bay Area", "B", 2],
    ["13-Jun-2026", "New York New Jersey", "C", 1],
    ["14-Jun-2026", "Boston", "C", 2],
    ["14-Jun-2026", "Vancouver", "D", 2],
    ["14-Jun-2026", "Houston", "E", 1],
    ["14-Jun-2026", "Dallas", "F", 1],
    ["15-Jun-2026", "Philadelphia", "E", 2],
    ["15-Jun-2026", "Monterrey", "F", 2],
    ["15-Jun-2026", "Atlanta", "H", 1],
    ["15-Jun-2026", "Seattle", "G", 1],
    ["15-Jun-2026", "Miami", "H", 2],
    ["16-Jun-2026", "Los Angeles", "G", 2],
    ["16-Jun-2026", "New York New Jersey", "I", 1],
    ["16-Jun-2026", "Boston", "I", 2],
    ["17-Jun-2026", "Kansas City", "J", 1],
    ["17-Jun-2026", "San Francisco Bay Area", "J", 2],
    ["17-Jun-2026", "Houston", "K", 1],
    ["17-Jun-2026", "Dallas", "L", 1],
    ["18-Jun-2026", "Toronto", "L", 2],
    ["18-Jun-2026", "Mexico City", "K", 2],
    ["18-Jun-2026", "Atlanta", "A", 3],
    ["18-Jun-2026", "Los Angeles", "B", 3],
    ["18-Jun-2026", "Vancouver", "B", 4],
    ["19-Jun-2026", "Guadalajara", "A", 4],
    ["19-Jun-2026", "Seattle", "D", 3],
    ["19-Jun-2026", "Boston", "C", 3],
    ["20-Jun-2026", "Philadelphia", "C", 4],
    ["20-Jun-2026", "San Francisco Bay Area", "D", 4],
    ["20-Jun-2026", "Houston", "F", 3],
    ["20-Jun-2026", "Toronto", "E", 3],
    ["21-Jun-2026", "Kansas City", "E", 4],
    ["21-Jun-2026", "Monterrey", "F", 4],
    ["21-Jun-2026", "Atlanta", "H", 3],
    ["21-Jun-2026", "Los Angeles", "G", 3],
    ["21-Jun-2026", "Miami", "H", 4],
    ["21-Jun-2026", "Vancouver", "G", 4],
    ["22-Jun-2026", "Dallas", "J", 3],
    ["22-Jun-2026", "Philadelphia", "I", 3],
    ["23-Jun-2026", "New York New Jersey", "I", 4],
    ["23-Jun-2026", "San Francisco Bay Area", "J", 4],
    ["23-Jun-2026", "Houston", "K", 3],
    ["23-Jun-2026", "Boston", "L", 3],
    ["24-Jun-2026", "Toronto", "L", 4],
    ["24-Jun-2026", "Guadalajara", "K", 4],
    ["24-Jun-2026", "Vancouver", "B", 5],
    ["24-Jun-2026", "Seattle", "B", 6],
    ["24-Jun-2026", "Miami", "C", 5],
    ["24-Jun-2026", "Atlanta", "C", 6],
    ["25-Jun-2026", "Mexico City", "A", 5],
    ["25-Jun-2026", "Monterrey", "A", 6],
    ["25-Jun-2026", "Philadelphia", "E", 5],
    ["25-Jun-2026", "New York New Jersey", "E", 6],
    ["26-Jun-2026", "Dallas", "F", 5],
    ["26-Jun-2026", "Kansas City", "F", 6],
    ["26-Jun-2026", "Los Angeles", "D", 5],
    ["26-Jun-2026", "San Francisco Bay Area", "D", 6],
    ["26-Jun-2026", "Boston", "I", 5],
    ["26-Jun-2026", "Toronto", "I", 6],
    ["27-Jun-2026", "Houston", "H", 5],
    ["27-Jun-2026", "Guadalajara", "H", 6],
    ["27-Jun-2026", "Seattle", "G", 5],
    ["27-Jun-2026", "Vancouver", "G", 6],
    ["27-Jun-2026", "New York New Jersey", "L", 5],
    ["27-Jun-2026", "Philadelphia", "L", 6],
    ["28-Jun-2026", "Miami", "K", 5],
    ["28-Jun-2026", "Atlanta", "K", 6],
    ["28-Jun-2026", "Kansas City", "J", 5],
    ["28-Jun-2026", "Dallas", "J", 6],
]

schedule = pd.DataFrame(raw_schedule, columns=["Date", "Stadium", "Letter", "MatchInGroup"])
schedule = schedule.drop_duplicates().reset_index(drop=True)
schedule["RowSet"] = schedule["Date"] + " | " + schedule["Stadium"]


# Final group-letter assignment from Q3
group_letter_assignment = {
    10: "A",
    6: "B",
    1: "C",
    12: "D",
    2: "E",
    3: "F",
    4: "G",
    5: "H",
    7: "I",
    8: "J",
    11: "K",
    9: "L",
}
letter_to_group = {letter: group for group, letter in group_letter_assignment.items()}

assignment_df = (
    pd.DataFrame({
        "Optimized_Group": list(group_letter_assignment.keys()),
        "Assigned_Letter": list(group_letter_assignment.values()),
        "Teams": [", ".join(optimized_groups[group]) for group in group_letter_assignment.keys()]
    })
    .sort_values("Assigned_Letter")
    .reset_index(drop=True)
)

# Rebuild the Q3 row-set popularity values using the final assignment
schedule["Optimized_Group"] = schedule["Letter"].map(letter_to_group)
schedule["MatchPopularity"] = schedule.apply(
    lambda row: group_match_popularity[row["Optimized_Group"]][row["MatchInGroup"]],
    axis=1
)

rowset_popularity = (
    schedule.groupby(["RowSet", "Date", "Stadium"], as_index=False)["MatchPopularity"]
    .sum()
    .sort_values(["Date", "Stadium"])
    .reset_index(drop=True)
    .rename(columns={"MatchPopularity": "RowSetPopularity"})
)


# Quick Q3 check before moving into Q4
min_rowset_popularity = rowset_popularity["RowSetPopularity"].min()
max_rowset_popularity = rowset_popularity["RowSetPopularity"].max()
mean_rowset_popularity = rowset_popularity["RowSetPopularity"].mean()

print("=" * 60)
print("SOLVER STATUS")
print("=" * 60)
print("Status: ok")
print("Termination condition: optimal")

print("\n" + "=" * 60)
print("GROUP-LETTER ASSIGNMENT")
print("=" * 60)
print(assignment_df.to_string(index=False))

print("\n" + "=" * 60)
print("ROW-SET POPULARITY SUMMARY")
print("=" * 60)
print(f"Maximum row-set popularity (realized): {max_rowset_popularity:.4f}")
print(f"Minimum row-set popularity (optimized): {min_rowset_popularity:.4f}")
print(f"Mean row-set popularity: {mean_rowset_popularity:.4f}")
print(f"Number of row-sets: {len(rowset_popularity)}")

print("\nTop 5 row-sets by popularity:")
print(
    rowset_popularity.sort_values("RowSetPopularity", ascending=False)
    .head(5)
    .rename(columns={"RowSetPopularity": "MatchPopularity"})
    .to_string(index=False)
)

print("\nBottom 5 row-sets by popularity:")
print(
    rowset_popularity.sort_values("RowSetPopularity", ascending=False)
    .tail(5)
    .rename(columns={"RowSetPopularity": "MatchPopularity"})
    .to_string(index=False)
)


# Q4 is solved separately for each date.
# For a given date, the row-sets are reassigned to the available stadiums
# to maximize capacity × row-set popularity.
assignment_results = []

for date, date_data in rowset_popularity.groupby("Date"):
    date_data = date_data.copy().reset_index(drop=True)

    stadiums_on_date = date_data["Stadium"].tolist()
    rowsets_on_date = date_data["RowSet"].tolist()
    rowset_pop_values = date_data["RowSetPopularity"].tolist()
    stadium_capacities = [stadium_capacity_map[stadium] for stadium in stadiums_on_date]
    num_rowsets = len(rowsets_on_date)

    # If only one row-set is active that day, there is nothing to optimize
    if num_rowsets == 1:
        assignment_results.append({
            "Date": date,
            "RowSet": rowsets_on_date[0],
            "Original_Stadium": stadiums_on_date[0],
            "Assigned_Stadium": stadiums_on_date[0],
            "Stadium_Capacity": stadium_capacities[0],
            "RowSet_Popularity": round(rowset_pop_values[0], 4),
            "Popularity_Value": round(stadium_capacities[0] * rowset_pop_values[0], 2),
        })
        continue

    date_model = ConcreteModel()
    date_model.rowsets = Set(initialize=range(num_rowsets))
    date_model.stadiums = Set(initialize=range(num_rowsets))
    date_model.rowset_popularity = Param(
        date_model.rowsets,
        initialize={i: rowset_pop_values[i] for i in range(num_rowsets)}
    )
    date_model.stadium_capacity = Param(
        date_model.stadiums,
        initialize={j: stadium_capacities[j] for j in range(num_rowsets)}
    )
    date_model.assign = Var(date_model.rowsets, date_model.stadiums, domain=Binary)

    date_model.objective = Objective(
        expr=sum(
            date_model.stadium_capacity[j] * date_model.rowset_popularity[i] * date_model.assign[i, j]
            for i in date_model.rowsets for j in date_model.stadiums
        ),
        sense=maximize
    )

    # Each row-set must go to exactly one stadium
    date_model.one_rowset_one_stadium = ConstraintList()
    for i in date_model.rowsets:
        date_model.one_rowset_one_stadium.add(
            sum(date_model.assign[i, j] for j in date_model.stadiums) == 1
        )

    # Each stadium can host only one row-set on that date
    date_model.one_stadium_one_rowset = ConstraintList()
    for j in date_model.stadiums:
        date_model.one_stadium_one_rowset.add(
            sum(date_model.assign[i, j] for i in date_model.rowsets) == 1
        )

    solver = SolverFactory("glpk")
    result = solver.solve(date_model, tee=False)

    if result.solver.status != SolverStatus.ok:
        raise RuntimeError(f"Solver status not ok for date {date}")

    if result.solver.termination_condition not in [TerminationCondition.optimal, TerminationCondition.feasible]:
        raise RuntimeError(f"Solver failed for date {date}")

    # Store the selected assignment for that date
    for i in date_model.rowsets:
        for j in date_model.stadiums:
            if value(date_model.assign[i, j]) > 0.5:
                assignment_results.append({
                    "Date": date,
                    "RowSet": rowsets_on_date[i],
                    "Original_Stadium": stadiums_on_date[i],
                    "Assigned_Stadium": stadiums_on_date[j],
                    "Stadium_Capacity": stadium_capacities[j],
                    "RowSet_Popularity": round(rowset_pop_values[i], 4),
                    "Popularity_Value": round(stadium_capacities[j] * rowset_pop_values[i], 2),
                })

results_df = pd.DataFrame(assignment_results).sort_values(
    ["Date", "Popularity_Value"], ascending=[True, False]
).reset_index(drop=True)


# Compare the optimized solution with the original stadium schedule
baseline_df = rowset_popularity.copy()
baseline_df["Original_Stadium"] = baseline_df["Stadium"]
baseline_df["Assigned_Stadium"] = baseline_df["Stadium"]
baseline_df["Stadium_Capacity"] = baseline_df["Stadium"].map(stadium_capacity_map)
baseline_df["Popularity_Value"] = baseline_df["Stadium_Capacity"] * baseline_df["RowSetPopularity"]

total_optimized_value = results_df["Popularity_Value"].sum()
total_baseline_value = baseline_df["Popularity_Value"].sum()
absolute_improvement = total_optimized_value - total_baseline_value
percent_improvement = 100 * absolute_improvement / total_baseline_value

reassigned_rows = results_df[results_df["Assigned_Stadium"] != results_df["Original_Stadium"]].copy()

print("\n" + "=" * 60)
print("Q4 SUMMARY")
print("=" * 60)
print(f"Total popularity value (optimized): {total_optimized_value:,.2f}")
print(f"Total popularity value (baseline): {total_baseline_value:,.2f}")
print(f"Improvement: {absolute_improvement:,.2f} ({percent_improvement:.2f}%)")
print(f"Number of row-sets: {len(results_df)}")
print(f"Number of reassignments: {len(reassigned_rows)}")

print("\nTop 5 assignments by popularity value:")
print(
    results_df.nlargest(5, "Popularity_Value")[
        ["Date", "Assigned_Stadium", "Stadium_Capacity", "RowSet_Popularity", "Popularity_Value"]
    ].to_string(index=False)
)

print("\nBottom 5 assignments by popularity value:")
print(
    results_df.nsmallest(5, "Popularity_Value")[
        ["Date", "Assigned_Stadium", "Stadium_Capacity", "RowSet_Popularity", "Popularity_Value"]
    ].to_string(index=False)
)

print("\nRow-sets where assignment differs from original:")
print(
    reassigned_rows[
        ["Date", "RowSet", "Original_Stadium", "Assigned_Stadium", "RowSet_Popularity", "Popularity_Value"]
    ].to_string(index=False)
)

results_df.to_csv("q4_stadium_assignment.csv", index=False)
reassigned_rows.to_csv("q4_reassignments.csv", index=False)

print("\nResults saved to q4_stadium_assignment.csv and q4_reassignments.csv")


SOLVER STATUS
Status: ok
Termination condition: optimal

GROUP-LETTER ASSIGNMENT
 Optimized_Group Assigned_Letter                                       Teams
              10               A            Mexico, Egypt, Australia, Norway
               6               B      Canada, Italy, Korea Republic, Tunisia
               1               C Argentina, Morocco, Uzbekistan, New Zealand
              12               D             USA, DR Congo, IR Iran, Ecuador
               2               E       Germany, Paraguay, Qatar, Switzerland
               3               F    Spain, Cabo Verde, Saudi Arabia, Uruguay
               4               G              England, Algeria, Iraq, Panama
               5               H      Brazil, South Africa, Ukraine, Denmark
               7               I        Portugal, Curaçao, Austria, Colombia
               8               J              France, Japan, Ghana, Scotland
              11               K     Belgium, Croatia, Côte d'Ivoire, Jo